<a href="https://colab.research.google.com/github/parkjeung/first-repository/blob/main/Streamlit_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ----------------------------------------------------------------------
# Page Configuration
# ----------------------------------------------------------------------
st.set_page_config(
    page_title="사용자 행동 대시보드",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# ----------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------
# Common data
monthly_labels = ['Sep 2016', 'Oct 2016', 'Nov 2016', 'Dec 2016', 'Jan 2017', 'Feb 2017', 'Mar 2017', 'Apr 2017', 'May 2017']
country_colors = {
    'United States': '#4285F4', 'India': '#DB4437', 'Vietnam': '#F4B400',
    'Turkey': '#0F9D58', 'Thailand': '#AB47BC', 'Brazil': '#00ACC1'
}
source_colors = {
    'Direct': '#4285F4', 'google': '#DB4437', 'youtube.com': '#F4B400',
    'referral': '#4285F4', 'organic': '#DB4437', 'affiliate': '#F4B400'
}

# KPI data
avg_dau = "2,274"
avg_mau = "62,391"
avg_stickiness = "3.6%"
avg_session_duration = "130"
avg_cart_conversion = "5.7%"

# Active User & Engagement data
mau_data = [58000, 60000, 98000, 65000, 48000, 50000, 58000, 50000, 48000]
dau_data = [2030, 2100, 3724, 2520, 1632, 1666, 2146, 1750, 1680]
stickiness_data = [(d / m) * 100 if m > 0 else 0 for d, m in zip(dau_data, mau_data)]
cart_conversion_data = [7.2, 6.2, 4.1, 3.4, 6.4, 5.5, 6.0, 5.5, 7.5]
session_duration_data = [140, 125, 110, 135, 128, 132, 130, 142, 145]

# AARRR Funnel data
aarrr_values = [75000, 48000, 25000, 15000, 4500]
aarrr_labels = ['획득 (방문)', '활성화 (상품조회)', '유지 (재방문)', '매출 (장바구니)', '매출 (구매완료)']

# Cohort data
cohort_data = {
    'Sep 2016': [100, 8.2, 6.5, 5.1, 4.3, 3.8, 3.1, 2.8, 2.5],
    'Oct 2016': [100, 7.9, 6.1, 4.9, 4.0, 3.5, 2.9, 2.6, None],
    'Nov 2016': [100, 9.1, 7.2, 5.8, 4.9, 4.2, 3.6, None, None],
    'Dec 2016': [100, 8.5, 6.8, 5.5, 4.6, 4.0, None, None, None],
    'Jan 2017': [100, 9.5, 7.8, 6.2, 5.1, None, None, None, None],
    'Feb 2017': [100, 8.8, 7.0, 5.7, None, None, None, None, None],
    'Mar 2017': [100, 9.2, 7.1, None, None, None, None, None, None],
    'Apr 2017': [100, 8.5, None, None, None, None, None, None, None],
    'May 2017': [100, None, None, None, None, None, None, None, None]
}
cohort_df = pd.DataFrame.from_dict(cohort_data, orient='index', columns=[f'M{i}' for i in range(9)])

# Country dataframes
countries = ['United States', 'India', 'Vietnam', 'Turkey', 'Thailand', 'Brazil']
country_mau_df = pd.DataFrame({
    'Month': monthly_labels,
    'United States': [18000, 19000, 21000, 20000, 18000, 19000, 20000, 19500, 20500],
    'India': [8000, 8500, 12000, 9000, 7000, 7500, 8000, 8200, 8100],
    'Vietnam': [5000, 5200, 7000, 6000, 5000, 5100, 5500, 5300, 5400],
    'Turkey': [6000, 6100, 8000, 7500, 6500, 6800, 7000, 7100, 7200],
    'Thailand': [7000, 7200, 9000, 8500, 7500, 7800, 8000, 8100, 8300],
    'Brazil': [9000, 9500, 11000, 10000, 9000, 9200, 9500, 9700, 9800]
}).melt(id_vars='Month', var_name='Country', value_name='MAU')

country_cart_df = pd.DataFrame({
    'Month': monthly_labels,
    'United States': [14, 13.5, 12, 13, 12.5, 11, 10, 10.5, 11],
    'India': [5, 4.5, 3, 4, 3.5, 2.5, 3, 3.5, 4],
    'Vietnam': [6, 5.5, 4, 5, 4.5, 3.5, 4, 4.5, 5],
    'Turkey': [4, 3.5, 2, 3, 2.5, 1.5, 2, 2.5, 3],
    'Thailand': [3, 2.5, 1, 2, 1.5, 0.5, 1, 1.5, 2],
    'Brazil': [5.5, 5, 3.5, 4.5, 4, 3, 3.5, 4, 4.5]
}).melt(id_vars='Month', var_name='Country', value_name='Cart Conversion')

# Traffic Source data
time_traffic_df = pd.DataFrame({
    'Month': monthly_labels,
    'Direct': [30, 25, 42, 42, 70, 68, 25, 30, 35],
    'google': [25, 40, 42, 43, 8, 5, 20, 52, 58],
    'youtube.com': [45, 35, 16, 55, 22, 15, 18, 18, 14]
})

country_traffic_df = pd.DataFrame({
    'Country': ['United States', 'India', 'Vietnam', 'Thailand', 'Turkey', 'Brazil'],
    'Direct': [176961, 14810, 3050, 16880, 16427, 13536],
    'YouTube': [83639, 13915, 941, 4167, 1378, 10567],
    'Google': [19896, 19918, 21671, 18412, 19587, 22204]
})

# SATV data
satv_data_list = []
satv_raw = {
    'Direct': [{'x': 2000, 'y': 18000, 'r': 25}, {'x': 2100, 'y': 18500, 'r': 26}, {'x': 2200, 'y': 19000, 'r': 24}, {'x': 2300, 'y': 19500, 'r': 27}, {'x': 2400, 'y': 20000, 'r': 28}, {'x': 2500, 'y': 20500, 'r': 29}, {'x': 2600, 'y': 21000, 'r': 30}, {'x': 2700, 'y': 21500, 'r': 31}, {'x': 2800, 'y': 22000, 'r': 32}],
    'google': [{'x': 10000, 'y': 15000, 'r': 18}, {'x': 11000, 'y': 16000, 'r': 19}, {'x': 12000, 'y': 15500, 'r': 17}, {'x': 13000, 'y': 17000, 'r': 20}, {'x': 14000, 'y': 18000, 'r': 21}, {'x': 15000, 'y': 19000, 'r': 22}, {'x': 16000, 'y': 20000, 'r': 23}, {'x': 17000, 'y': 21000, 'r': 24}, {'x': 18000, 'y': 22000, 'r': 25}],
    'youtube.com': [{'x': 1500, 'y': 8000, 'r': 10}, {'x': 5000, 'y': 40000, 'r': 5}, {'x': 6000, 'y': 45000, 'r': 4}, {'x': 2000, 'y': 9000, 'r': 9}, {'x': 2200, 'y': 9500, 'r': 11}, {'x': 2300, 'y': 10000, 'r': 12}, {'x': 2400, 'y': 10500, 'r': 13}, {'x': 2500, 'y': 11000, 'r': 14}, {'x': 2600, 'y': 11500, 'r': 15}]
}
for channel, data in satv_raw.items():
    for i, point in enumerate(data):
        satv_data_list.append({'Channel': channel, 'Spend': point['x'], 'Acquisition': point['y'], 'LTV': point['r'], 'Month': monthly_labels[i]})
satv_df = pd.DataFrame(satv_data_list)


# ----------------------------------------------------------------------
# CSS Styling
# ----------------------------------------------------------------------
st.markdown("""
<style>
    .kpi-card {
        background-color: #FFFFFF;
        border-radius: 0.75rem;
        box-shadow: 0 4px 6px -1px rgb(0 0 0 / 0.1), 0 2px 4px -2px rgb(0 0 0 / 0.1);
        padding: 1.5rem;
        border: 1px solid #E2E8F0;
    }
    .kpi-title {
        color: #005F73;
        font-weight: 500;
        font-size: 1rem;
        margin-bottom: 0.5rem;
    }
    .kpi-value {
        color: #0A9396;
        font-weight: 700;
        font-size: 2.5rem;
    }
    .kpi-subtext {
        color: #6c757d;
        font-size: 0.875rem;
        margin-top: 0.25rem;
    }
    .stRadio > label {
        font-size: 0.9rem !important;
    }
</style>
""", unsafe_allow_html=True)

# ----------------------------------------------------------------------
# Header
# ----------------------------------------------------------------------
st.title("📊 사용자 행동 대시보드")
st.markdown("주요 지표 및 사용자 행동 트렌드")

# ----------------------------------------------------------------------
# KPI Section
# ----------------------------------------------------------------------
st.markdown("### 주요 지표 요약")
kpi_cols = st.columns(5)
with kpi_cols[0]:
    st.markdown(f"""
    <div class="kpi-card">
        <div class="kpi-title">DAU (평균)</div>
        <div class="kpi-value">{avg_dau}</div>
        <div class="kpi-subtext">일별 고유 사용자 (평균)</div>
    </div>
    """, unsafe_allow_html=True)
with kpi_cols[1]:
    st.markdown(f"""
    <div class="kpi-card">
        <div class="kpi-title">MAU (평균)</div>
        <div class="kpi-value">{avg_mau}</div>
        <div class="kpi-subtext">월별 고유 사용자 (평균)</div>
    </div>
    """, unsafe_allow_html=True)
with kpi_cols[2]:
    st.markdown(f"""
    <div class="kpi-card">
        <div class="kpi-title">고착도</div>
        <div class="kpi-value">{avg_stickiness}</div>
        <div class="kpi-subtext">DAU / MAU (평균)</div>
    </div>
    """, unsafe_allow_html=True)
with kpi_cols[3]:
    st.markdown(f"""
    <div class="kpi-card">
        <div class="kpi-title">평균 체류시간 (초)</div>
        <div class="kpi-value">{avg_session_duration}</div>
        <div class="kpi-subtext">체류시간 (평균)</div>
    </div>
    """, unsafe_allow_html=True)
with kpi_cols[4]:
    st.markdown(f"""
    <div class="kpi-card">
        <div class="kpi-title">일간 카트 전환율</div>
        <div class="kpi-value">{avg_cart_conversion}</div>
        <div class="kpi-subtext">장바구니 담은 유저수 / DAU (평균)</div>
    </div>
    """, unsafe_allow_html=True)

st.divider()

# ----------------------------------------------------------------------
# Chart Section 1: Active Users & Engagement
# ----------------------------------------------------------------------
row1_col1, row1_col2 = st.columns(2)

with row1_col1:
    st.subheader("기간별 활성 사용자 & 고착도")
    active_user_metric = st.radio("지표 선택", ['월간(MAU)', '일간(DAU)', '고착도'], key='active_user', horizontal=True)

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    # MAU Trace
    fig.add_trace(go.Scatter(x=monthly_labels, y=mau_data, name='MAU', mode='lines',
                             line=dict(color='#4285F4'), visible=(active_user_metric == '월간(MAU)')), secondary_y=False)
    # DAU Trace
    fig.add_trace(go.Scatter(x=monthly_labels, y=dau_data, name='DAU', mode='lines',
                             line=dict(color='#DB4437'), visible=(active_user_metric == '일간(DAU)')), secondary_y=False)
    # Stickiness Trace
    fig.add_trace(go.Scatter(x=monthly_labels, y=stickiness_data, name='고착도', mode='lines',
                             line=dict(color='#0F9D58'), visible=(active_user_metric == '고착도')), secondary_y=True)

    fig.update_layout(height=400, margin=dict(l=20, r=20, t=40, b=20), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    fig.update_yaxes(title_text="사용자 수", secondary_y=False)
    fig.update_yaxes(title_text="고착도 (%)", secondary_y=True, visible=(active_user_metric == '고착도'))
    st.plotly_chart(fig, use_container_width=True)

with row1_col2:
    st.subheader("기간별 체류시간 & 카트 전환율")
    engagement_metric = st.radio("지표 선택", ['평균 체류시간', '카트 전환율'], key='engagement', horizontal=True)

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    # Session Duration Trace
    fig.add_trace(go.Scatter(x=monthly_labels, y=session_duration_data, name='평균 체류시간', mode='lines',
                             line=dict(color='#0A9396'), visible=(engagement_metric == '평균 체류시간')), secondary_y=False)
    # Cart Conversion Trace
    fig.add_trace(go.Scatter(x=monthly_labels, y=cart_conversion_data, name='카트 전환율', mode='lines',
                             line=dict(color='#CA6702'), visible=(engagement_metric == '카트 전환율')), secondary_y=True)

    fig.update_layout(height=400, margin=dict(l=20, r=20, t=40, b=20), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    fig.update_yaxes(title_text="평균 체류시간 (초)", secondary_y=False, visible=(engagement_metric == '평균 체류시간'))
    fig.update_yaxes(title_text="카트 전환율 (%)", secondary_y=True, visible=(engagement_metric == '카트 전환율'))
    st.plotly_chart(fig, use_container_width=True)

# ----------------------------------------------------------------------
# Chart Section 2: Funnel & Cohort
# ----------------------------------------------------------------------
row2_col1, row2_col2 = st.columns(2)

with row2_col1:
    st.subheader("AARRR 퍼널 분석")
    fig = go.Figure(go.Funnel(
        y=aarrr_labels,
        x=aarrr_values,
        textposition="inside",
        textinfo="value+percent previous",
        marker={"color": ['#003f5c', '#444e86', '#955196', '#dd5182', '#ff6e54']}
    ))
    fig.update_layout(height=400, margin=dict(l=20, r=20, t=40, b=20))
    st.plotly_chart(fig, use_container_width=True)

with row2_col2:
    st.subheader("코호트 분석 (사용자 유지율)")
    fig = go.Figure(data=go.Heatmap(
        z=cohort_df.values,
        x=cohort_df.columns,
        y=cohort_df.index,
        colorscale='Teal',
        text=cohort_df.applymap(lambda x: f'{x:.1f}%' if pd.notnull(x) else ''),
        texttemplate="%{text}",
        showscale=False
    ))
    fig.update_layout(height=400, margin=dict(l=20, r=20, t=40, b=20), yaxis_autorange='reversed')
    st.plotly_chart(fig, use_container_width=True)

# ----------------------------------------------------------------------
# Chart Section 3: Country Analysis
# ----------------------------------------------------------------------
row3_col1, row3_col2 = st.columns(2)

with row3_col1:
    st.subheader("국가별 월간 활성 사용자(MAU)")
    fig = px.line(country_mau_df, x='Month', y='MAU', color='Country',
                  color_discrete_map=country_colors, markers=True)
    fig.update_layout(height=400, margin=dict(l=20, r=20, t=40, b=20), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    st.plotly_chart(fig, use_container_width=True)

with row3_col2:
    st.subheader("국가별 카트 전환율")
    fig = px.line(country_cart_df, x='Month', y='Cart Conversion', color='Country',
                  color_discrete_map=country_colors, markers=True)
    fig.update_layout(height=400, margin=dict(l=20, r=20, t=40, b=20), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    fig.update_yaxes(title_text="카트 전환율 (%)")
    st.plotly_chart(fig, use_container_width=True)

# ----------------------------------------------------------------------
# Chart Section 4: Traffic Source Analysis
# ----------------------------------------------------------------------
row4_col1, row4_col2 = st.columns(2)

with row4_col1:
    st.subheader("기간별 Traffic Source Top3")
    fig = px.line(time_traffic_df, x='Month', y=['Direct', 'google', 'youtube.com'],
                  color_discrete_map=source_colors, markers=True)
    fig.update_layout(height=400, margin=dict(l=20, r=20, t=40, b=20), legend=dict(title='Source', orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    fig.update_yaxes(title_text="비중 (%)")
    st.plotly_chart(fig, use_container_width=True)

with row4_col2:
    st.subheader("국가별 Traffic Source Top3")
    fig = px.bar(country_traffic_df, x='Country', y=['Direct', 'YouTube', 'Google'],
                 color_discrete_map=source_colors, barmode='group')
    fig.update_layout(height=400, margin=dict(l=20, r=20, t=40, b=20), legend=dict(title='Source', orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    fig.update_yaxes(title_text="Sessions")
    st.plotly_chart(fig, use_container_width=True)

# ----------------------------------------------------------------------
# Chart Section 5: SATV Analysis
# ----------------------------------------------------------------------
st.subheader("SATV 분석 (채널 효율성)")
fig = px.scatter(
    satv_df,
    x="Spend",
    y="Acquisition",
    size="LTV",
    color="Channel",
    hover_name="Month",
    color_discrete_map=source_colors,
    animation_frame="Month",
    animation_group="Channel",
    size_max=60,
    range_x=[1000, 20000],
    range_y=[5000, 50000],
    title="월별 채널별 비용, 신규 사용자, LTV"
)
fig.update_layout(height=500, margin=dict(l=20, r=20, t=40, b=20))
st.plotly_chart(fig, use_container_width=True)

st.divider()

# ----------------------------------------------------------------------
# Conclusion Section
# ----------------------------------------------------------------------
st.header("종합 결론 및 인사이트")
st.markdown("""
<div style="background-color:#F8F9FA; border-left: 5px solid #005F73; padding: 1rem; border-radius: 5px;">
    <h3 style="color:#005F73;">핵심 분석 요약</h3>
    <p>AARRR 퍼널 분석 결과, <strong>획득(Acquisition)</strong> 단계에서는 특히 10-11월 YouTube 채널을 통해 대규모 트래픽이 발생했으나, 이후 <strong>활성화(Activation)</strong> 및 <strong>유지(Retention)</strong> 단계에서 급격한 이탈이 확인됩니다. 이는 코호트 분석에서 나타난 낮은 초기 리텐션과 일치하는 결과입니다. <strong>SATV 분석</strong> 역시 YouTube 채널이 낮은 비용으로 많은 사용자(Acquisition)를 데려왔지만, 이들의 LTV(Value)는 매우 낮아 질적으로 비효율적이었음을 보여줍니다. 즉, 현재 비즈니스는 '밑 빠진 독에 물 붓기' 형태로, 신규 사용자 유입에만 집중하고 이들을 붙잡아두지 못하는 구조적 문제를 안고 있습니다. 반면, 미국/캐나다 등 고성과 그룹은 모든 퍼널 단계에서 건실한 전환율을 보여주고 있어 질적 성장을 견인하고 있습니다.</p>

    <h3 style="color:#005F73; margin-top:1.5rem;">전략적 제언</h3>
    <ul>
        <li>
            <strong>AARRR 퍼널 개선 전략:</strong>
            <ul>
                <li><strong>활성화(Activation) 개선:</strong> 신규 방문자가 첫 방문 시 긍정적인 경험을 하도록 온보딩 프로세스를 강화하고, 개인화된 상품 추천 기능을 도입하여 '상품 조회' 전환율을 높여야 합니다.</li>
                <li><strong>유지(Retention) 강화:</strong> 낮은 리텐션은 가장 시급한 문제입니다. 첫 구매 고객 대상 재방문 쿠폰, 관심 상품 재입고 알림, 이메일/앱 푸시 등 CRM 활동을 통해 재방문율과 재구매율을 높여야 합니다.</li>
            </ul>
        </li>
        <li>
            <strong>고성과 지역 재확대 (미국/캐나다):</strong>
            <ul>
                <li><strong>타겟팅 강화 및 예산 재분배:</strong> 구매력이 검증된 미국과 캐나다를 핵심 타겟으로 확립하고 마케팅 예산을 집중합니다.</li>
                <li><strong>충성 고객 관리 (Referral 유도):</strong> 구매력이 높은 기존 고객 대상 리워드, VIP 혜택을 강화하고 '친구 추천' 프로그램을 도입하여 바이럴 마케팅 효과(추천)를 유도합니다.</li>
            </ul>
        </li>
        <li>
            <strong>저성과 지역 맞춤형 공략 (동남아/터키):</strong>
            <ul>
                <li><strong>YouTube 플랫폼 질적 성장 목표 활용:</strong> 양적 유입이 확인된 YouTube를 활용하되, 단순 노출이 아닌 '회원가입', '뉴스레터 구독' 등 명확한 활성화(Activation) 목표를 설정하고 LTV가 높은 타겟에 집중합니다.</li>
                <li><strong>철저한 현지화(Localization):</strong> 국가별 문화, 소비 패턴, 주요 결제 방식을 분석하여 제품 페이지와 결제 프로세스를 현지화하고, UI/UX를 최적화하여 구매 전환 장벽을 낮춥니다.</li>
            </ul>
        </li>
    </ul>
</div>
""", unsafe_allow_html=True)